# Complete Plot Reference Guide
## All plot types that have been asked or could be asked in CSIS3754/3764 exams

## Setup — Run this cell first

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import learning_curve, KFold
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.naive_bayes import GaussianNB

# Generate sample data for demonstrations
np.random.seed(42)
df = pd.DataFrame({
    'Age':        np.random.randint(20, 80, 200),
    'Income':     np.random.choice(['Low','Medium','High','Very High'], 200),
    'Education':  np.random.choice(['Uneducated','Graduate','Doctorate'], 200),
    'Gender':     np.random.choice(['Male','Female'], 200),
    'BloodType':  np.random.choice(['A+','A-','B+','B-','O+','O-','AB+','AB-'], 200),
    'Score':      np.random.uniform(0, 100, 200),
    'Billing':    np.random.uniform(1000, 50000, 200),
    'Result':     np.random.choice(['Normal','Abnormal','Inconclusive'], 200)
})
print('Sample data created:', df.shape)

---
## 1. PIE CHART
**When asked:** Class balance, counterfeit vs genuine, existing vs attrited customers, test result distribution

In [ ]:
counts = df['Result'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    counts,
    labels=counts.index,
    autopct=lambda p: f'{p:.1f}%',   # change decimal places as required
    startangle=90
)
plt.title('Test Result Distribution')
plt.show()
print(counts)

---
## 2. BAR CHART / COUNT PLOT
**When asked:** Number of records per category, card category per education level

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Education', hue='Result')
plt.title('Result Count by Education Level')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 3. SCATTER PLOT — Simple
**When asked:** Relationship between two numeric features e.g. Credit Limit vs Avg_Open_To_Buy

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Age', y='Billing')
plt.title('Age vs Billing Amount')
plt.tight_layout()
plt.show()

---
## 4. SCATTER PLOT — Combined (with hue)
**When asked:** Income vs Credit Limit for different Card Categories, Age vs Na_K ratio per medication

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='Income', y='Billing', hue='Gender')
plt.title('Income vs Billing by Gender')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 5. SCATTER PLOT — Clusters with centroids
**When asked:** Visualise k-means clusters on scatter plots with centroid markers

In [ ]:
# Generate cluster data
X_cluster, _ = make_blobs(n_samples=200, centers=3, random_state=42)
cluster_df = pd.DataFrame(X_cluster, columns=['Feature1', 'Feature2'])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_df['Cluster'] = kmeans.fit_predict(X_cluster)

palette = sns.color_palette('tab10', n_colors=3)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=cluster_df, x='Feature1', y='Feature2',
                hue='Cluster', palette=palette, alpha=0.7)

# Plot centroids as X markers
for i, centre in enumerate(kmeans.cluster_centers_):
    plt.scatter(centre[0], centre[1],
                color=palette[i], marker='X', s=200,
                edgecolors='black', linewidths=1.5,
                label=f'Centroid {i}')

plt.title('Clusters with Centroids')
plt.tight_layout()
plt.show()

---
## 6. HEATMAP — Correlation matrix
**When asked:** Correlation between features, correlation with target variable

In [ ]:
numeric_df = df[['Age', 'Score', 'Billing']]
corr_matrix = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5
)
plt.title('Correlation Matrix Heatmap')
plt.tight_layout()
plt.show()
print(corr_matrix)

---
## 7. BOXPLOT
**When asked:** Detecting outliers in numeric features e.g. Weight, Length, Width, Thickness

In [ ]:
numeric_features = ['Age', 'Score', 'Billing']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, numeric_features):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

---
## 8. HISTOGRAM
**When asked:** Distribution of a numeric feature, age distribution, billing distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='Age', bins=20, kde=True)
plt.title('Age Distribution')
plt.tight_layout()
plt.show()

---
## 9. LEARNING CURVE
**When asked:** Draw learning curve for each classifier, visualise overfitting/underfitting

In [ ]:
from sklearn.model_selection import learning_curve

def plot_learning_curve(model, name, X, y, cv):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=cv, scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, label='Training score', color='blue')
    plt.fill_between(train_sizes, train_mean-train_std,
                     train_mean+train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean, label='CV score', color='orange')
    plt.fill_between(train_sizes, val_mean-val_std,
                     val_mean+val_std, alpha=0.1, color='orange')
    plt.title(f'Learning Curve — {name}')
    plt.xlabel('Training Size')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Example usage
X_lc, y_lc = make_classification(n_samples=500, random_state=42)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
plot_learning_curve(GaussianNB(), 'Naive Bayes', X_lc, y_lc, kfold)

---
## 10. ELBOW CURVE
**When asked:** Determine optimal k for k-means clustering

In [ ]:
X_km, _ = make_blobs(n_samples=300, centers=4, random_state=42)
k_range  = range(2, 11)
inertia  = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_km)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o', color='blue')
plt.title('Elbow Curve')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

---
## 11. SILHOUETTE SCORE PLOT
**When asked:** Determine optimal k alongside elbow curve

In [ ]:
sil_scores = []

for k in k_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_km)
    sil_scores.append(silhouette_score(X_km, labels))

plt.figure(figsize=(8, 5))
plt.plot(k_range, sil_scores, marker='o', color='orange')
plt.title('Silhouette Score per k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

best_k = list(k_range)[sil_scores.index(max(sil_scores))]
print(f'Best k: {best_k} (score = {max(sil_scores):.4f})')

---
## 12. CONFUSION MATRIX HEATMAP
**When asked:** Visualise model predictions vs actual labels

In [ ]:
# Simulate confusion matrix
y_true = np.random.choice([0, 1, 2], 200)
y_pred = np.random.choice([0, 1, 2], 200)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## 13. PCA SCATTER PLOT
**When asked:** Visualise clusters after reducing dimensions with PCA

In [ ]:
X_pca, _ = make_blobs(n_samples=300, centers=3, n_features=4, random_state=42)

pca    = PCA(n_components=2)
X_2d   = pca.fit_transform(X_pca)

kmeans  = KMeans(n_clusters=3, random_state=42, n_init=10)
labels  = kmeans.fit_predict(X_pca)

pca_df            = pd.DataFrame(X_2d, columns=['PC1', 'PC2'])
pca_df['Cluster'] = labels

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2',
                hue='Cluster', palette='tab10', s=60, alpha=0.8)
plt.title('PCA Scatter Plot — Clusters')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.show()

print(f'Total variance retained: {pca.explained_variance_ratio_.sum()*100:.2f}%')

---
## 14. LINE PLOT
**When asked:** Trends over time, admissions per date

In [ ]:
dates = pd.date_range('2023-01-01', periods=30)
admissions = np.random.randint(5, 25, 30)

plt.figure(figsize=(12, 5))
plt.plot(dates, admissions, marker='o', color='blue')
plt.title('Daily Admissions')
plt.xlabel('Date')
plt.ylabel('Number of Admissions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 15. CATEGORY PLOT (catplot)
**When asked:** Number of customers per Card Category for each Education Level

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Education', hue='Income')
plt.title('Income Category by Education Level')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## QUICK REFERENCE SUMMARY


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                    PLOT QUICK REFERENCE                         ║
╠══════════════════════════════════════════════════════════════════╣
║ Plot               │ When to use                                ║
╠══════════════════════════════════════════════════════════════════╣
║ plt.pie()          │ Class balance, distributions               ║
║ sns.countplot()    │ Category counts, card types per education  ║
║ sns.scatterplot()  │ Relationship between two features          ║
║ sns.scatterplot()  │ With hue= for grouped/cluster scatter      ║
║ sns.heatmap()      │ Correlation matrix, confusion matrix       ║
║ sns.boxplot()      │ Outlier detection                          ║
║ sns.histplot()     │ Feature distribution                       ║
║ learning_curve()   │ Overfitting/underfitting visualisation     ║
║ plt.plot(inertia)  │ Elbow curve for optimal k                  ║
║ plt.plot(sil)      │ Silhouette score for optimal k             ║
║ PCA + scatter      │ Visualise clusters in 2D                   ║
║ plt.plot(dates)    │ Trends over time                           ║
╠══════════════════════════════════════════════════════════════════╣
║ AUTOPCT FORMATS:                                                 ║
║ 0 decimals: autopct=lambda p: f'{p:.0f}%'                       ║
║ 1 decimal:  autopct=lambda p: f'{p:.1f}%'                       ║
║ 2 decimals: autopct=lambda p: f'{p:.2f}%'                       ║
║ 3 decimals: autopct=lambda p: f'{p:.3f}%'                       ║
╚══════════════════════════════════════════════════════════════════╝
""")